# Initial setup

In [1]:
import sqlite3, pandas as pd, time, random, string
from datetime import datetime, timedelta

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

def q(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params or {})

def execmany(sql, rows):
    cur.executemany(sql, rows)
    conn.commit()

def execsql(sql):
    cur.executescript(sql)
    conn.commit()

# Tables & Constraints

In [2]:
execsql("""
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers(
  customer_id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT UNIQUE
);

CREATE TABLE orders(
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  amount REAL NOT NULL CHECK(amount >= 0),
  status TEXT NOT NULL CHECK(status IN ('NEW','PAID','SHIPPED','CANCELLED')),
  FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);
""")

execmany("INSERT INTO customers(customer_id,name,email) VALUES(?,?,?)", [
    (1,"Alice","alice@acme.com"),
    (2,"Bob","bob@acme.com"),
])

execmany("INSERT INTO orders(order_id,customer_id,order_date,amount,status) VALUES(?,?,?,?,?)", [
    (100,1,"2026-02-01",120.0,"PAID"),
    (101,1,"2026-02-03", 60.5,"NEW"),
    (102,2,"2026-02-05", 12.0,"SHIPPED"),
])

q("SELECT * FROM customers")

,customer_id,name,email
0,1,Alice,alice@acme.com
1,2,Bob,bob@acme.com


# Breaking constraints

# Tasks
	*	For each error: write which constraint was violated and why.
	*	Bonus: change the schema so email is optional but still unique when provided.

In [3]:
# 1) NOT NULL violation
cur.execute("INSERT INTO customers(customer_id,name,email) VALUES(?,?,?)", (3,None,"x@x.com"))
conn.commit()

IntegrityError: NOT NULL constraint failed: customers.name

In [4]:
# 2) UNIQUE violation
cur.execute("INSERT INTO customers(customer_id,name,email) VALUES(?,?,?)", (3,"Eve","alice@acme.com"))
conn.commit()

IntegrityError: UNIQUE constraint failed: customers.email

In [5]:
# 3) CHECK violation
cur.execute("INSERT INTO orders(order_id,customer_id,order_date,amount,status) VALUES(?,?,?,?,?)",
            (103,1,"2026-02-06",-1,"NEW"))
conn.commit()

IntegrityError: CHECK constraint failed: amount >= 0

In [6]:
# 4) FK violation
cur.execute("INSERT INTO orders(order_id,customer_id,order_date,amount,status) VALUES(?,?,?,?,?)",
            (104,999,"2026-02-06",10,"NEW"))
conn.commit()

IntegrityError: FOREIGN KEY constraint failed

# Joins

# Tasks
	*	Change to INNER JOIN. What changes?
	*	Add a filter: only status IN ('PAID','SHIPPED')
	*	Show customers with 0 paid orders (hint: LEFT JOIN + WHERE on aggregated result).

In [7]:
q("""
SELECT c.customer_id, c.name,
       COUNT(o.order_id) AS orders,
       SUM(o.amount) AS revenue
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY 1,2
ORDER BY revenue DESC;
""")

,customer_id,name,orders,revenue
0,1,Alice,2,180.5
1,2,Bob,1,12.0


# Indexing + Query plan

# Tasks
	1.	Compare query plan output before vs after (look for SCAN vs SEARCH).
	2.	Repeat with a compound filter: WHERE customer_id=? AND order_date=? and try a composite index.



In [8]:
execsql("DROP TABLE IF EXISTS big_orders;")
execsql("""
CREATE TABLE big_orders(
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  amount REAL NOT NULL
);
""")

# generate 200k rows
rows = []
start = datetime(2024,1,1)
for i in range(1,200_001):
    cid = random.randint(1, 5000)
    d = (start + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d")
    amt = round(random.random()*500,2)
    rows.append((i,cid,d,amt))

execmany("INSERT INTO big_orders(order_id,customer_id,order_date,amount) VALUES(?,?,?,?)", rows)
print("rows:", q("SELECT COUNT(*) AS n FROM big_orders")["n"][0])

rows: 200000


Check query plan + timing

In [9]:
target_customer = 42

print(q("EXPLAIN QUERY PLAN SELECT * FROM big_orders WHERE customer_id = :c", {"c": target_customer}))
t0 = time.time()
q("SELECT * FROM big_orders WHERE customer_id = :c", {"c": target_customer})
print("seconds:", time.time()-t0)

   id  parent  notused           detail
0   2       0      216  SCAN big_orders
seconds: 0.010136604309082031


Add index

In [10]:
execsql("CREATE INDEX idx_big_orders_customer ON big_orders(customer_id);")

print(q("EXPLAIN QUERY PLAN SELECT * FROM big_orders WHERE customer_id = :c", {"c": target_customer}))
t0 = time.time()
q("SELECT * FROM big_orders WHERE customer_id = :c", {"c": target_customer})
print("seconds:", time.time()-t0)

   id  parent  notused                                             detail
0   3       0       61  SEARCH big_orders USING INDEX idx_big_orders_c...
seconds: 0.0009067058563232422


# Transactions + ACID thinking

# Tasks
	•	Explain which ACID property you observed.
	•	Implement a correct transfer function that checks balances and commits only if valid.

In [11]:
execsql("""
DROP TABLE IF EXISTS accounts;
CREATE TABLE accounts(
  account_id INTEGER PRIMARY KEY,
  owner TEXT NOT NULL,
  balance REAL NOT NULL CHECK(balance >= 0)
);

INSERT INTO accounts VALUES (1,'Alice',100.0);
INSERT INTO accounts VALUES (2,'Bob',  50.0);
""")

q("SELECT * FROM accounts")

,account_id,owner,balance
0,1,Alice,100.0
1,2,Bob,50.0


In [12]:
try:
    conn.execute("BEGIN;")
    conn.execute("UPDATE accounts SET balance = balance - 80 WHERE account_id=1;")
    # force a failure: Bob would go negative if we subtract too much OR violate CHECK using manual set
    conn.execute("UPDATE accounts SET balance = -1 WHERE account_id=2;")  # violates CHECK
    conn.execute("COMMIT;")
except Exception as e:
    conn.execute("ROLLBACK;")
    print("Rolled back because:", e)

q("SELECT * FROM accounts")

Rolled back because: CHECK constraint failed: balance >= 0


,account_id,owner,balance
0,1,Alice,100.0
1,2,Bob,50.0


# Views vs Materialized Views

SQLite has Views. It doesn’t have native materialized views, but we can simulate with a table + refresh.

# Tasks
	•	Insert a new paid order and show how VIEW reflects it immediately, while “MV” needs refresh.

In [13]:
execsql("""
DROP VIEW IF EXISTS v_customer_revenue;
CREATE VIEW v_customer_revenue AS
SELECT customer_id, SUM(amount) AS revenue
FROM orders
GROUP BY customer_id;
""")
q("SELECT * FROM v_customer_revenue")

,customer_id,revenue
0,1,180.5
1,2,12.0


In [14]:
# Simmulated MV

execsql("""
DROP TABLE IF EXISTS mv_customer_revenue;
CREATE TABLE mv_customer_revenue AS
SELECT customer_id, SUM(amount) AS revenue
FROM orders
GROUP BY customer_id;
""")
q("SELECT * FROM mv_customer_revenue")

,customer_id,revenue
0,1,180.5
1,2,12.0


In [15]:
# Now wee need to refresh the MV

execsql("""
DELETE FROM mv_customer_revenue;
INSERT INTO mv_customer_revenue
SELECT customer_id, SUM(amount)
FROM orders
GROUP BY customer_id;
""")

# Change Data Capture

# Time based CDC example

In [16]:
execsql("""
DROP TABLE IF EXISTS products;
CREATE TABLE products(
  product_id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  price REAL NOT NULL,
  last_updated TEXT NOT NULL
);
""")

now = datetime.now().isoformat(timespec="seconds")
execmany("INSERT INTO products(product_id,name,price,last_updated) VALUES(?,?,?,?)", [
    (1,"A",10.0, now),
    (2,"B",20.0, now),
])

In [17]:
time.sleep(1)
t1 = datetime.now().isoformat(timespec="seconds")
conn.execute("UPDATE products SET price=12.0, last_updated=? WHERE product_id=1", (t1,))
conn.commit()

last_checkpoint = now
q("SELECT * FROM products WHERE last_updated > :cp", {"cp": last_checkpoint})

,product_id,name,price,last_updated
0,1,A,12.0,2026-02-23T17:28:56


Sidenote: Deletes won't be captured

▶ Add a is_deleted column and implement “soft delete”.

# Trigger based CDC example

In [18]:
execsql("""
DROP TABLE IF EXISTS products_audit;

CREATE TABLE products_audit(
  audit_id INTEGER PRIMARY KEY AUTOINCREMENT,
  op TEXT NOT NULL,
  product_id INTEGER,
  old_price REAL,
  new_price REAL,
  change_time TEXT NOT NULL
);

DROP TRIGGER IF EXISTS trg_products_update;

CREATE TRIGGER trg_products_update
AFTER UPDATE ON products
BEGIN
  INSERT INTO products_audit(op, product_id, old_price, new_price, change_time)
  VALUES ('U', OLD.product_id, OLD.price, NEW.price, datetime('now'));
END;
""")

conn.execute("UPDATE products SET price=13.5, last_updated=datetime('now') WHERE product_id=1")
conn.commit()
q("SELECT * FROM products_audit ORDER BY audit_id DESC LIMIT 5;")

,audit_id,op,product_id,old_price,new_price,change_time
0,1,U,1,12.0,13.5,2026-02-23 16:29:00
